In [1]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-y420u4ko
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-y420u4ko
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=93f1b686421f41da0dc26608a27d11f19b7dda633c5ade777007eff774f29d83
  Stored in directory: /tmp/pip-ephem-wheel-cache-_a4laz_g/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:

!mkdir -p ./checkpoints

!cp /kaggle/input/models/tejaspkalkar/latest-vit-b-dim2-dist-arc-122/pytorch/default/1/latest_vit_b_dim2_dist_arc.pth ./checkpoints/

In [3]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from tqdm import tqdm
import clip

# ==========================================
# 1. NaN-Safe DistArc Loss Module
# ==========================================
class DistArcLoss(nn.Module):
    def __init__(self, in_features, out_features, m=0.4, lam=0.005, radial_gap=0.5):
        super(DistArcLoss, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.m = m
        self.lam = lam
        
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        
        self.W = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.kaiming_uniform_(self.W)

        radii = np.arange(2.0, 2.0 + out_features * radial_gap, radial_gap).astype('float32')
        self.register_buffer('r', torch.tensor(radii))

    def forward(self, x, labels):
        batch_size = x.size(0)
        
        x_norm = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.W, p=2, dim=0)
        
        cosine = torch.matmul(x_norm, W_norm).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        target_logit = cosine[torch.arange(batch_size), labels]
        
        sine = torch.sqrt((1.0 - torch.pow(target_logit, 2)).clamp(min=1e-7))
        marginal_cosine = target_logit * self.cos_m - sine * self.sin_m 

        W_scaled = W_norm * self.r
        target_w_r = W_scaled[:, labels].T 
        R = -target_w_r + x 
        
        R_norm = F.normalize(R, p=2, dim=1)
        target_w_r_norm = F.normalize(-target_w_r, p=2, dim=1)
        cos_phi = torch.sum(R_norm * target_w_r_norm, dim=1) 

        x_sq = torch.sum(x**2, dim=1, keepdim=True) 
        w_r_sq = torch.sum(W_scaled**2, dim=0, keepdim=True) 
        dist_sq = x_sq + w_r_sq - 2 * torch.matmul(x, W_scaled) 

        num_exp = marginal_cosine + cos_phi - (self.lam * dist_sq[torch.arange(batch_size), labels])
        
        denom_terms = cosine - (self.lam * dist_sq)
        denom_terms[torch.arange(batch_size), labels] = marginal_cosine
        
        loss = -num_exp + torch.logsumexp(denom_terms, dim=1)
        return loss.mean()

    def predict(self, x):
        W_scaled = F.normalize(self.W, p=2, dim=0) * self.r
        x_sq = torch.sum(x**2, dim=1, keepdim=True)
        w_r_sq = torch.sum(W_scaled**2, dim=0, keepdim=True)
        dist_sq = x_sq + w_r_sq - 2 * torch.matmul(x, W_scaled)
        return torch.argmin(dist_sq, dim=1)

# ==========================================
# 2. ViT-B Model Wrapper
# ==========================================
class ViT_B_HyperSpaceX(nn.Module):
    def __init__(self, embedding_size=2, num_classes=100, loss_type='dist_arc', device='cuda'):
        super(ViT_B_HyperSpaceX, self).__init__()
        
        self.clip_model, self.preprocess = clip.load("ViT-B/32", device=device)
        self.clip_model = self.clip_model.float() 
        self.visual = self.clip_model.visual
        
        clip_out_dim = 512 
        self.projection = nn.Linear(clip_out_dim, embedding_size)
        self.loss_type = loss_type
        
        if self.loss_type == 'cross_entropy':
            self.classifier = nn.Linear(embedding_size, num_classes)
            
    def forward(self, x):
        x = self.visual(x) 
        embeddings = self.projection(x)
        
        if self.loss_type == 'cross_entropy':
            return self.classifier(embeddings)
        return embeddings

# ==========================================
# 3. Data Loading
# ==========================================
def get_dataloaders(preprocess, batch_size):
    train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, transform=preprocess, download=True)
    val_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, transform=preprocess, download=True)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    return train_loader, val_loader

# ==========================================
# 4. Main Training Loop
# ==========================================
def train_model(
    embedding_size=2, 
    loss_type='dist_arc', 
    epochs=150, 
    batch_size=128, 
    lr=1e-4,        
    weight_decay=5e-4, 
    lam=0.005,
    save_dir='./checkpoints'
):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_classes = 100 
    
    os.makedirs(save_dir, exist_ok=True)
    
    best_save_path = os.path.join(save_dir, f'best_vit_b_dim{embedding_size}_{loss_type}.pth')
    latest_save_path = os.path.join(save_dir, f'latest_vit_b_dim{embedding_size}_{loss_type}.pth')

    print(f"--- Starting Training ---")
    print(f"Model: ViT-B (CLIP) | Emb Size: {embedding_size} | Loss: {loss_type} | Target: CIFAR-100")

    model = ViT_B_HyperSpaceX(embedding_size, num_classes, loss_type, device)

    preprocess_fn = model.preprocess
    train_loader, val_loader = get_dataloaders(preprocess_fn, batch_size)

    if torch.cuda.device_count() > 1:
        print(f"Awesome, utilizing {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    model = model.to(device)
    
    if loss_type == 'dist_arc':
        criterion = DistArcLoss(embedding_size, num_classes, lam=lam).to(device)
        optimizer = optim.SGD([
            {'params': model.parameters()},
            {'params': criterion.parameters()}
        ], lr=lr, weight_decay=weight_decay, momentum=0.9)
    else:
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9)

    best_acc = 0.0
    start_epoch = 0

    # --- RESUME LOGIC ---
    if os.path.exists(latest_save_path):
        print(f"Found existing checkpoint! Loading from {latest_save_path}...")
        checkpoint = torch.load(latest_save_path)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if loss_type == 'dist_arc' and 'criterion_state_dict' in checkpoint:
            criterion.load_state_dict(checkpoint['criterion_state_dict'])
            
        start_epoch = checkpoint['epoch']
        best_acc = checkpoint['best_acc']
        print(f"Resuming training from Epoch {start_epoch + 1} with Best Acc so far: {best_acc:.2f}%")

    # --- EPOCH LOOP ---
    for epoch in range(start_epoch, epochs):
        
        model.train()
        if loss_type == 'dist_arc': criterion.train()
        
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            pbar.set_postfix({'loss': f"{running_loss/len(train_loader):.4f}"})
            
        model.eval()
        if loss_type == 'dist_arc': criterion.eval()
        
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                
                if loss_type == 'cross_entropy':
                    _, predicted = torch.max(outputs.data, 1)
                else:
                    predicted = criterion.predict(outputs)
                    
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        acc = 100 * correct / total
        print(f"Validation Accuracy: {acc:.2f}%")
        
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_acc': best_acc,
            'loss_type': loss_type,
            'embedding_size': embedding_size
        }
        if loss_type == 'dist_arc':
            checkpoint['criterion_state_dict'] = criterion.state_dict()
            
        torch.save(checkpoint, latest_save_path)
        
        if acc > best_acc:
            best_acc = acc
            print(f"--> New Best Accuracy! ({best_acc:.2f}%) Saving BEST model to {best_save_path}")
            torch.save(checkpoint, best_save_path)

if __name__ == '__main__':
    # Changed total epochs to 245!
    train_model(embedding_size=2, loss_type='dist_arc', epochs=245)

--- Starting Training ---
Model: ViT-B (CLIP) | Emb Size: 2 | Loss: dist_arc | Target: CIFAR-100


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 132MiB/s]
100%|██████████| 169M/169M [00:02<00:00, 60.6MB/s]


Awesome, utilizing 2 GPUs!
Found existing checkpoint! Loading from ./checkpoints/latest_vit_b_dim2_dist_arc.pth...
Resuming training from Epoch 123 with Best Acc so far: 69.62%


Epoch 123/245 [Val]: 100%|██████████| 79/79 [00:24<00:00,  3.19it/s]


Validation Accuracy: 70.03%
--> New Best Accuracy! (70.03%) Saving BEST model to ./checkpoints/best_vit_b_dim2_dist_arc.pth


...

Validation Accuracy: 71.51%


Epoch 244/245 [Val]: 100%|██████████| 79/79 [00:25<00:00,  3.12it/s]


Validation Accuracy: 70.86%


Epoch 245/245 [Val]: 100%|██████████| 79/79 [00:25<00:00,  3.10it/s]


Validation Accuracy: 66.56%
